In [8]:
import json
import os
import zipfile
import pandas as pd
import kagglehub

print("Downloading the full Spider dataset via kagglehub...")
dataset_dir = kagglehub.dataset_download("jeromeblanchet/yale-universitys-spider-10-nlp-dataset")

nested_zip = os.path.join(dataset_dir, "spider.zip")
extract_path = "./spider_unpacked"

if os.path.exists(nested_zip):
    with zipfile.ZipFile(nested_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    extract_path = dataset_dir

# Search for BOTH train and dev files
spider_files = ["train_spider.json", "dev.json"]
original_spider = []

for root, dirs, files in os.walk(extract_path):
    for f_name in spider_files:
        if f_name in files:
            full_p = os.path.join(root, f_name)
            print(f"Loading metadata from: {full_p}")
            with open(full_p, "r", encoding="utf-8") as f:
                original_spider.extend(json.load(f))

print(f"\nCombined Reference Pool Built! Total Spider records loaded: {len(original_spider)}")

Using Colab cache for faster access to the 'yale-universitys-spider-10-nlp-dataset' dataset.
Loading metadata from: /kaggle/input/yale-universitys-spider-10-nlp-dataset/spider/train_spider.json
Loading metadata from: /kaggle/input/yale-universitys-spider-10-nlp-dataset/spider/dev.json

Combined Reference Pool Built! Total Spider records loaded: 8034


In [9]:
cleaned_json_path = "/hinglish_textosql_cleaned.json"
output_json_path = "hinglish_textosql_final.json"
output_csv_path = "hinglish_textosql_final.csv"

with open(cleaned_json_path, "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

print(f"Successfully loaded local cleaned dataset.")
print(f"-> Total row records detected: {len(eval_dataset)}")
print(f"-> Expected variations per row record: 3 (Light, Natural, Heavy)")
print(f"-> Initialized target file assets for mapping step.")

Successfully loaded local cleaned dataset.
-> Total row records detected: 1468
-> Expected variations per row record: 3 (Light, Natural, Heavy)
-> Initialized target file assets for mapping step.


In [10]:
from tqdm import tqdm
import json
import pandas as pd
import re

def get_sql_tokens(sql_str):
    if not isinstance(sql_str, str):
        return set()
    tokens = re.findall(r'\b\w+\b', sql_str.lower())
    return set(tokens)

# Build query mapping pool across ALL Spider records
spider_pool = []
for item in original_spider:
    spider_query = item.get("query") or item.get("sql")
    if isinstance(spider_query, str):
        spider_pool.append({
            "db_id": item["db_id"],
            "tokens": get_sql_tokens(spider_query),
            "raw_sql": spider_query
        })

final_dataset = []
mapped_count = 0
unmapped_rows = 0

for row in tqdm(eval_dataset, desc="Executing Global Token Matcher"):
    golden_sql = row.get('sql', '')
    row_tokens = get_sql_tokens(golden_sql)

    best_db_id = None
    best_match_score = 0.0

    for spider_item in spider_pool:
        intersection = row_tokens.intersection(spider_item["tokens"])
        union = row_tokens.union(spider_item["tokens"])

        if union:
            jaccard_score = len(intersection) / len(union)
            if jaccard_score > best_match_score and jaccard_score > 0.82:
                best_match_score = jaccard_score
                best_db_id = spider_item["db_id"]

    if best_db_id:
        compiled_record = {
            "db_id": best_db_id,
            "english": row.get("english", ""),
            "hinglish_light": row.get("hinglish_light", ""),
            "hinglish_natural": row.get("hinglish_natural", ""),
            "hinglish_hindi_heavy": row.get("hinglish_hindi_heavy", ""),
            "sql": golden_sql
        }
        final_dataset.append(compiled_record)
        mapped_count += 1
    else:
        unmapped_rows += 1

with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=4, ensure_ascii=False)

final_df = pd.DataFrame(final_dataset)
final_df.to_csv(output_csv_path, index=False, encoding="utf-8")

print(f"\nGlobal Alignment Pipeline Completed!")
print(f"-> Successfully mapped rows: {mapped_count} / {len(eval_dataset)}")
print(f"-> Unmapped/Skipped rows: {unmapped_rows}")

Executing Global Token Matcher: 100%|██████████| 1468/1468 [00:27<00:00, 53.07it/s]



Global Alignment Pipeline Completed!
-> Successfully mapped rows: 1465 / 1468
-> Unmapped/Skipped rows: 3


In [11]:
import os
from google.colab import files

print(f"JSON File Path: '{output_json_path}' ({os.path.getsize(output_json_path) / 1024:.2f} KB)")
print(f"CSV File Path:  '{output_csv_path}' ({os.path.getsize(output_csv_path) / 1024:.2f} KB)")

files.download(output_json_path)

files.download(output_csv_path)

print("\nDownload requests sent successfully!")

JSON File Path: 'hinglish_textosql_final.json' (837.24 KB)
CSV File Path:  'hinglish_textosql_final.csv' (602.14 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download requests sent successfully!
